# Hierarchical models and testing
# Haoyang Qian
# 2026/09/20
---

### This parametric model can the effectiveness of the intervention, estimate the model parameters and test if the intervention has an effect or not.

In [1]:
#%pip install pandas numpy

In [2]:
#%pip install bambi

In [3]:
import os
os.environ["PYTENSOR_FLAGS"] = "cxx="

In [4]:
import pandas as pd
import numpy as np
import bambi as bmb
import arviz as az

In [5]:
# read in data
data_file = "towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers or yes/no

# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 diferent studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set 
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')

## Model & Hypothesis

**Model Specification:**
- **Likelihood:** $y_{i,j} \sim \text{Binomial}(N_{i,j}, p_{i,j})$ for study $j \in \{1,\dots,7\}$ and group $i \in \{\text{control}, \text{social}\}$.
- **Link Function & Hierarchical Structure:** 
  $$\text{logit}(p_{i,j}) = \alpha + a_j + (\beta + b_j) \cdot \text{IsSocial}_{i,j}$$
  - $\beta$: Fixed intervention effect (primary quantity of interest).
  - $a_j \sim \mathcal{N}(0, \tau_a^2)$: Random intercept for baseline study variation.
  - $b_j \sim \mathcal{N}(0, \tau_b^2)$: Random slope for between-study heterogeneity in intervention effect.

**Hypothesis Formulation:**
- $H_0: \beta \le 0$ (Social norm intervention is ineffective or harmful)
- $H_1: \beta > 0$ (Social norm intervention increases towel reuse)

Print out dataframe to see data structure.

In [6]:
print(combined_data)

    reuse  total    group study
0      74    211  control     1
1     103    277  control     2
2      77    135  control     3
3      82    187  control     4
4      21     25  control     5
5     123    147  control     6
6      28     30  control     7
7      98    222   social     1
8     587   1318   social     2
9     406    655   social     3
10    278    555   social     4
11     21     24   social     5
12    472    576   social     6
13    101    132   social     7


In [7]:
model = bmb.Model(
    "prop(reuse, total) ~ group + (group | study)", 
    family="binomial",
    data=combined_data
)

MCMC sampling

In [9]:
idata = model.fit(draws=2000, tune=1000, chains=4, random_seed=1)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, group, 1|study_sigma, 1|study_offset, group|study_sigma, group|study_offset]


C:\Users\12532\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 10 seconds.
There were 44 divergences after tuning. Increase `target_accept` or reparameterize.


In [10]:
summary_90 = az.summary(
    idata,
    var_names=["group", "1|study_sigma", "group|study_sigma"],
    ci_prob=0.90,
)


Print out summary of the parameter values by the posterior mean and 90% probability interval

In [11]:
print(summary_90)

                            mean     sd eti90_lb eti90_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
group[social]              0.177  0.123   -0.045     0.36     1983     1538  1.00    0.0033  0.0038
1|study_sigma               1.16    0.4     0.66      1.9     1768     1469  1.00    0.0098  0.0083
group|study_sigma[social]   0.18   0.18    0.011     0.53     1707     1445  1.00    0.0055  0.0089


In [12]:
beta_samples = idata.posterior["group"].sel(group_dim="social").values.flatten()

post_prob = np.mean(beta_samples > 0)

or_samples = np.exp(beta_samples)
or_mean = np.mean(or_samples)

or_hdi = az.hdi(or_samples, prob=0.90)

sigma_mean = summary_90.loc["group|study_sigma[social]", "mean"]
print(f"P(beta_social > 0 | data): {post_prob:.3f}")
print(f"Odds Ratio: {or_mean:.3f}")
print(f"Odds Ratio 90% HDI: [{or_hdi[0]:.3f}, {or_hdi[1]:.3f}]")
print(f"Random slope variation: {sigma_mean:.3f}")

P(beta_social > 0 | data): 0.924
Odds Ratio: 1.203
Odds Ratio 90% HDI: [0.977, 1.450]
Random slope variation: 0.178


## Conclusion

* **Hypothesis Test:** $P(\beta_{\text{social}} > 0 \mid \text{Data}) = {0.924}$, rejecting $H_0$. Descriptive social norm messaging significantly increases towel reuse.
* **Effect Size:** Posterior mean Odds Ratio (OR) = 1.203 (90% HDI: [0.977, 1.450]). Since the HDI is strictly above 1, the intervention effectively boosts reuse odds.
* **Heterogeneity:** The random slope variation (`group|study_sigma`=0.178) confirms study-level differences, validating the hierarchical model structure.